In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['MLFLOW_TRACKING_URI'] = os.getenv('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME'] = os.getenv('MLFLOW_TRACKING_USERNAME')
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.getenv('MLFLOW_TRACKING_PASSWORD')

In [ ]:
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\DataScienceProject'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_file_name: Path
    all_params: dict
    target_column: str
    mlflow_uri: str

In [5]:
from src.datascience.constants import *
from src.datascience.utils.main_utils import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config= self.config.model_evaluation
        params= self.params.ElasticNet
        schema= self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config= ModelEvaluationConfig(
            root_dir= config.root_dir,
            test_data_path= config.test_data_path,
            model_path= config.model_path,
            all_params= params,
            metric_file_name= config.metric_file_name,
            target_column= schema.name,
            mlflow_uri= os.getenv('MLFLOW_TRACKING_URI')
        )
        return model_evaluation_config

[ 2026-05-12 17:54:59,145 : INFO : __init__ : Logger initialized successfully ]


In [6]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import joblib
from urllib.parse import urlparse
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig) -> None:
        self.config = config
    
    def eval_metrics(self, actual, pred):
        r2 = r2_score(actual, pred)
        mae = mean_absolute_error(actual, pred)
        mse = mean_squared_error(actual, pred)
        rmse = np.sqrt(mse)
        return r2, mae, mse, rmse
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop(columns= [self.config.target_column])
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        mlflow.set_experiment("datascience-project")
        tracking_uri_type_score = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted = model.predict(test_x)
            (r2, mae, mse, rmse) = self.eval_metrics(test_y, predicted)

            ## Saving metrics as local
            scores = {"r2" : r2, "MAE" : mae, "MSE" : mse, "RMSE" : rmse}

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("R2", r2)
            mlflow.log_metric("MAE", mae)
            mlflow.log_metric("MSE", mse)
            mlflow.log_metric("RMSE", rmse)

            if tracking_uri_type_score != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

c:\Users\singh\OneDrive\Desktop\DataScienceProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config= model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[ 2026-05-12 17:55:08,234 : INFO : main_utils : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-12 17:55:08,237 : INFO : main_utils : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-12 17:55:08,242 : INFO : main_utils : YAML file: schema.yaml Loaded Successfully ]
[ 2026-05-12 17:55:08,247 : INFO : main_utils : Create Directory at: artifacts ]
[ 2026-05-12 17:55:08,250 : INFO : main_utils : Create Directory at: artifacts/model_evaluation ]


2026/05/12 17:55:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 17:55:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\singh\OneDrive\Desktop\DataScienceProject
2026/05/12 17:55:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/12 17:55:16 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\singh\OneDrive\Desktop\DataScienceProject
2026/05/12 17:55:16 INFO mlflow.utils.environment: Detected uv project at c:\Users\singh\OneDrive\Desktop\DataScienceProject. Attempting to export requirements via 'uv export'.
202

🏃 View run flawless-wolf-716 at: https://dagshub.com/navneetsxngh/DataScienceProject.mlflow/#/experiments/0/runs/676bde53390d4e9b8af783b6e68244aa
🧪 View experiment at: https://dagshub.com/navneetsxngh/DataScienceProject.mlflow/#/experiments/0
